# Manifest Tool Registration via CLI

This notebook shows how to define a manifest with a custom tool and register it using the `agentic` CLI.

In [ ]:
import os, subprocess, textwrap, json
from pathlib import Path
from getpass import getpass

if not os.getenv("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass("Enter OPENAI_API_KEY (press Enter to skip): ")

# Write a minimal manifest with a custom HTTP tool and a single agent using it.
manifest_path = Path("examples/tool_manifest.yaml")
manifest_path.write_text(textwrap.dedent(
    """
    name: http-demo
    description: Simple HTTP tool demo
    tools:
      http-get:
        name: http-get
        type: http.get
        config:
          timeout: 5
    agents:
      fetcher:
        name: fetcher
        role: fetches data
        tools: [http-get]
    steps:
      - name: fetch
        agent: fetcher
        description: fetch data via http-get
    """
))
manifest_path

In [ ]:
# Register the manifest into a local registry directory using the CLI.
registry_dir = Path("registry")
registry_dir.mkdir(exist_ok=True)
result = subprocess.run(
    ["agentic", "register", "--workflow", str(manifest_path), "--registry", str(registry_dir)],
    capture_output=True,
    text=True,
)
print("CLI exit code:", result.returncode)
print(result.stdout)
registry_dir.glob("*.yaml")

In [ ]:
# Inspect the registered manifest content
registered = registry_dir / manifest_path.name.replace(".yaml", ".yaml")
print(registered.read_text())